# Implementing a Learning Rate Schedule

This demonstration is available as a Jupyter notebook or julia script
[here](https://github.com/FluxML/MLJFlux.jl/tree/dev/docs/src/common_workflows/learning_rate_schedules).

A **learning rate schedule** describes dynamic changes to the learning rate as training
proceeds.  In MLJ such schedules are realized using MLJ's `IteratedModel` wrapper. Here
we demonstrate exponential decay, step decay, and cyclic learning rates.

In this illustration, no attempt has been made to optimize the schedule parameters.

In [1]:
using Pkg
PKG_ENV = joinpath(@__DIR__, "..", "..", "..")
Pkg.activate(PKG_ENV);
Pkg.instantiate();

  Activating project at `~/GoogleDrive/Julia/MLJ/MLJFlux/docs`


**Julia version** is assumed to be at least 1.12.

### Basic Imports

In [2]:
using MLJ               # Has MLJFlux models
using Flux              # For more flexibility
import Optimisers       # native Flux.jl optimisers no longer supported
using StableRNGs        # for reproducibility across Julia versions

stable_rng() = StableRNGs.StableRNG(123);

### Loading and Splitting the Data

In [3]:
iris = load_iris() # a named-tuple of vectors
y, X = unpack(iris, ==(:target), rng=stable_rng())
X = fmap(column-> Float32.(column), X) # Flux prefers Float32 data
(X_train, X_test), (y_train, y_test) = partition(
    (X, y), 0.8,
    multi = true,
    shuffle = true,
    rng=stable_rng(),
);

### Instantiating the model

Now let's construct our model, as we did for Incremental Training with
MLJFlux:

In [4]:
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux
clf = NeuralNetworkClassifier(
    builder=MLJFlux.MLP(; hidden=(5,4), σ=Flux.relu),
    eta = 0.05,
    batch_size=8,
    epochs=10,
    rng=stable_rng(),
)

[ Info: For silent loading, specify `verbosity=0`. 
import MLJFlux ✔


NeuralNetworkClassifier(
  builder = MLP(
        hidden = (5, 4), 
        σ = NNlib.relu), 
  finaliser = NNlib.softmax, 
  optimiser = Adam(eta=0.001, beta=(0.9, 0.999), epsilon=1.0e-8), 
  eta = 0.05, 
  loss = Flux.Losses.crossentropy, 
  epochs = 10, 
  batch_size = 8, 
  lambda = 0.0, 
  alpha = 0.0, 
  rng = StableRNGs.LehmerRNG(state=0x000000000000000000000000000000f7), 
  optimiser_changes_trigger_retraining = false, 
  acceleration = CPU1{Nothing}(nothing), 
  embedding_dims = Dict{Symbol, Real}())

To implement the learning rate schedule, we wrap our model using MLJ's
[`IteratedModel`](https://juliaai.github.io/MLJ.jl/dev/controlling_iterative_models/#Controlling-Iterative-Models)
wrapper, including an appropriate `WithModelDo` callback control. Since we are using the
wrapper, we might as well add early stopping criteria.

In [5]:
decay(factor) = WithModelDo(model -> (model.eta *= factor))
controls = [
    Step(1),                        # train one epoch for each control cycle
    decay(0.95),                    # reduce learning rate by scaling by 0.95 each cycle
    Patience(3),                    # Stop when the loss drops below 0.001
    WithLossDo(loss -> @info "loss: $loss"),       # print the loss
    InvalidValue(),                 # Stop if NaN or Inf encountered
    NumberLimit(100),               # Stop after 100 control cycles
]
iterated_clf = IteratedModel(clf; controls, resampling=nothing);

We use `resampling=nothing` here to signal that all supplied data should be used for
training and that the training loss (as reported by the model) should be the basis for
the stopping criteria. To use an out-of-sample loss instead, specify
`resampling=Holdout()`.

In [6]:
mach = machine(iterated_clf, X_train, y_train)
fit!(mach);

[ Info: Training machine(ProbabilisticIteratedModel(model = NeuralNetworkClassifier(builder = MLP(hidden = (5, 4), …), …), …), …).
[ Info: No iteration parameter specified. Using `iteration_parameter=:(epochs)`. 
[ Info: loss: 0.8926081
[ Info: loss: 0.49345642
[ Info: loss: 0.338484
[ Info: loss: 0.1856992
[ Info: loss: 0.11490267
[ Info: loss: 0.0843781
[ Info: loss: 0.070317164
[ Info: loss: 0.06344699
[ Info: loss: 0.05891313
[ Info: loss: 0.054841064
[ Info: loss: 0.05116478
[ Info: loss: 0.047917776
[ Info: loss: 0.0462549
[ Info: loss: 0.04923024
[ Info: loss: 0.05809458
[ Info: loss: 0.068200484
[ Info: final loss: 0.068200484
[ Info: final training loss: 0.068200484
[ Info: Stop triggered by EarlyStopping.Patience(3) stopping criterion. 
[ Info: Total of 16 iterations. 


We can predict using the wrapped model as we would the original model, as here:

In [7]:
test_loss = cross_entropy(predict(mach, X_test), y_test)

0.43126359583891904

To get step decay, we can use the `skip` control wrapper:

In [8]:
step_decay(factor, width) = MLJIteration.skip(decay(factor), predicate=width);

For example, `step_decay(0.7, 10)` will multiply the learning rate by 0.7 every 10
epochs:

In [9]:
controls = [
    Step(1),                        # train one epoch for each control cycle
    step_decay(0.7, 10),
    Patience(3),                    # Stop when the loss drops below 0.001
    WithLossDo(loss -> @info "loss: $loss"),       # print the loss
    InvalidValue(),                 # Stop if NaN or Inf encountered
    NumberLimit(100),               # Stop after 100 control cycles
]
iterated_clf = IteratedModel(clf; controls, resampling=nothing)
mach = machine(iterated_clf, X_train, y_train) |> fit!;

[ Info: Training machine(ProbabilisticIteratedModel(model = NeuralNetworkClassifier(builder = MLP(hidden = (5, 4), …), …), …), …).
[ Info: No iteration parameter specified. Using `iteration_parameter=:(epochs)`. 
[ Info: loss: 0.8926081
[ Info: loss: 0.4949374
[ Info: loss: 0.33910838
[ Info: loss: 0.20393783
[ Info: loss: 0.102364086
[ Info: loss: 0.07725816
[ Info: loss: 0.08481228
[ Info: loss: 0.06428369
[ Info: loss: 0.06818254
[ Info: loss: 0.06204281
[ Info: loss: 0.044060886
[ Info: loss: 0.04729958
[ Info: loss: 0.048344623
[ Info: loss: 0.04804867
[ Info: loss: 0.04797266
[ Info: loss: 0.04850576
[ Info: loss: 0.04913268
[ Info: loss: 0.049667485
[ Info: final loss: 0.049667485
[ Info: final training loss: 0.049667485
[ Info: Stop triggered by EarlyStopping.Patience(3) stopping criterion. 
[ Info: Total of 18 iterations. 


In [10]:
test_loss = cross_entropy(predict(mach, X_test), y_test)

0.34018196032888626

While cyclic learning rates can be implemented using a more complicated variation on the
above pattern, a built-in control is provided for convenience. Consult the
`CycleLearningRate` document string for details and some differences to be found in
other implementations.

In [11]:
cycle = CycleLearningRate(stepsize=4, lower=0.001, upper=0.05, learning_rate_parameter=:eta)
controls = [
    Step(1),                        # train one epoch for each control cycle
    cycle,
    Patience(3),                    # Stop when the loss drops below 0.001
    WithLossDo(loss -> @info "loss: $loss"),       # print the loss
    InvalidValue(),                 # Stop if NaN or Inf encountered
    NumberLimit(100),               # Stop after 100 control cycles
]
iterated_clf = IteratedModel(clf; controls, resampling=nothing)
mach = machine(iterated_clf, X_train, y_train) |> fit!;

[ Info: Training machine(ProbabilisticIteratedModel(model = NeuralNetworkClassifier(builder = MLP(hidden = (5, 4), …), …), …), …).
[ Info: No iteration parameter specified. Using `iteration_parameter=:(epochs)`. 
[ Info: loss: 0.8926081
[ Info: loss: 0.5643743
[ Info: loss: 0.5103982
[ Info: loss: 0.41017845
[ Info: loss: 0.2641572
[ Info: loss: 0.2630486
[ Info: loss: 0.10116409
[ Info: loss: 0.0703797
[ Info: loss: 0.07176808
[ Info: loss: 0.051934265
[ Info: loss: 0.053149268
[ Info: loss: 0.06296733
[ Info: loss: 0.07152607
[ Info: final loss: 0.07152607
[ Info: final training loss: 0.07152607
[ Info: Stop triggered by EarlyStopping.Patience(3) stopping criterion. 
[ Info: Total of 13 iterations. 


In [12]:
test_loss = cross_entropy(predict(mach, X_test), y_test)

0.21526962150821466

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*